### Importing Libraries

In [ ]:
import numpy as np 
import h5py
import matplotlib.colors as pltColors
import matplotlib.pyplot as plt
import scipy.io
import os
import re
from utils import *
from scipy.optimize import curve_fit
enable_underscore_cleanup() # Sets ipython hook to delete user defined varibles that start with _ at the end of each cell execution

### Loading data

In [ ]:
###############################################
runNumbers = [21]
folder = '/sdf/data/lcls/ds/cxi/cxil1037623/scratch/davidjr/dg2ipmReproc/'
###############################################
# (1) keys_to_combine: some keys loaded for each shot & stored per shot 
# (2) keys_to_sum: some keys loaded per each run and added 
# (3) keys_to_check : check if some keys exits and have same values in all runs and load these keys 
_keys_to_combine = [#'CXI-DG2-BMMON-WF/ROI_area',
                    'jungfrau4M/azav_mask0_azav', # Unfiltered
                   'jungfrau4M/azav_mask1_azav', # Filtered
                   'dg2ipmReproc/sum',
                   'dg2ipmReproc/peaks',
                   'dg2ipmReproc/xpos',
                   'dg2ipmReproc/ypos',
                   'gas_detector/f_11_ENRC',
                   'ebeam/photon_energy',
                   'evr/code_183',
                   'evr/code_137',
                   'evr/code_141',
                   'lightStatus/xray',
                  'jungfrau4M/Full_thres_sum',
                  'feeBld/hproj',
                  'lightStatus/laser',
                   'ipm_dg2/sum',
                   'unixTime',
                   'epicsUser/gasCell_pressure',
                  ]

_keys_to_sum = ['Sums/jungfrau4M_calib_xrayOn_thresADU1']
#               'Sums/jungfrau4M_calib_thresADU1']

_keys_to_check = ['UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_q',
                'UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_q',
                 'UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_idxq',
                 'UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_idxq',
                'UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_qbin',
                'UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_qbin',
                'UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_qbins',
                'UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_qbins',
                'UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_userMask',
                'UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_userMask',
                'UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_matrix_q', # This are only needed once
                'UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_matrix_phi', # This are only needed once
                'UserDataCfg/jungfrau4M/x',
                'UserDataCfg/jungfrau4M/y',
                'UserDataCfg/jungfrau4M/z',
                'UserDataCfg/jungfrau4M/cmask']
# Load the data in
_data = combineRuns(runNumbers, folder, _keys_to_combine, _keys_to_sum, _keys_to_check, verbose=False)  # this is the function to load the data with defined keys

# Filtered Data
azavFiltered = np.squeeze(_data['jungfrau4M/azav_mask0_azav']) # I(q) : 1D azimuthal average of signals in each q bin
qbinFiltered = _data['UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_qbin'] # q bin-size
qFiltered = _data['UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_q'] # q bins 
qbinsFiltered = _data['UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_qbins'] # q bins
userMaskFiltered = _data['UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_userMask'].astype(bool) # User mask for this region
qbinSizeFiltered = np.bincount(_data['UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_idxq'])
# Unfiltered Data
azav = np.squeeze(_data['jungfrau4M/azav_mask1_azav']) # I(q) : 1D azimuthal average of signals in each q bin
qbin = _data['UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_qbin'] # q bin-size
q = _data['UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_q'] # q bins 
qbins = _data['UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_qbins'] # q bins
userMask = _data['UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_userMask'].astype(bool) # User mask for this region
qbinSize = np.bincount(_data['UserDataCfg/jungfrau4M/azav_mask1__azav_mask1_idxq'])
# Other Data
matrix_q = _data['UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_matrix_q'].reshape(8,512,1024) # Q values J4M shaped
matrix_phi = _data['UserDataCfg/jungfrau4M/azav_mask0__azav_mask0_matrix_phi'].reshape(8,512,1024) # phi valyes J4M shaped
xrayOn = _data['evr/code_137'].astype(bool)  # xray on events
xrayOn2 = _data['lightStatus/xray'].astype(bool)  # xray on events
laserOn = _data['lightStatus/laser'].astype(bool)  # xray on events
jungfrau_sum = _data['Sums/jungfrau4M_calib_xrayOn_thresADU1']  # Total Jungfrau detector counts summed in a run
#jungfrau_sum = data['Sums/jungfrau4M_calib_thresADU1']   # Total Jungfrau detector counts with Thresholds added, summed in a run 
x = _data['UserDataCfg/jungfrau4M/x'] # coordinates of Jungfrau detector x,y,z
y = _data['UserDataCfg/jungfrau4M/y']
z = _data['UserDataCfg/jungfrau4M/z'] 

cmask = _data['UserDataCfg/jungfrau4M/cmask'].astype(bool) # Mask for detector created 
run_indicator = _data['run_indicator'] # run indicator for each shot
#pressure = data['epicsAll/gasCell_pressure']  # pressure in gas cell
xray_energy = _data['gas_detector/f_11_ENRC']   # xray energy from gas detector (not calibrated to actual values)
xray_eV = _data['ebeam/photon_energy']    # x-ray energy energy in eV|
numPhotons = _data['jungfrau4M/Full_thres_sum']
spec = _data['feeBld/hproj'] # Shot to shot spectrometer
#dg2traces = data['CXI-DG2-BMMON-WF/ROI_area'] # Downsampling by a factor of 8, makes the refitting much better in the long run
gasPressure = _data['epicsUser/gasCell_pressure']
eventTime = _data['unixTime']
ipm = _data['dg2ipmReproc/sum']
ipmpeaks = _data['dg2ipmReproc/peaks']
xpos = _data['dg2ipmReproc/xpos']
ypos = _data['dg2ipmReproc/ypos']
#dg2tracesFull = data['CXI-DG2-BMMON-WF/ROI_rebin_data']

eventTime = eventTime[1:-1];
gasPressure = gasPressure[1:-1];
## Loading spectrometer calibration
spec_calib = np.load('ZnCalib/15-28.npy')

### Finding the input photon energy from the center of the average of the spectrometer readings

In [ ]:
# Define a Gaussian function
def _gaussian(x, a, x0, sigma, d):
    return a * np.exp(-(x - x0)**2 / (2 * sigma**2)) + d

# Generate some fake data for demonstration (you already have your actual data)
specAverage = spec.mean(axis=0)

# Initial guess
_initial_guess = [2.5e6, 9.64, 0.1, 10]

# Bounds: ([min_a, min_x0, min_sigma, min_d], [max_a, max_x0, max_sigma, max_d])
_bounds = ([0.5e6, 9.0, 0.01, -np.inf], [3e6, 10.5, 1.0, np.inf])

# Fit the data with bounds
_popt, _pcov = curve_fit(_gaussian, spec_calib, specAverage, p0=_initial_guess, bounds=_bounds, maxfev=10000)

# Extract and print fitted parameters
_a_fit, photon_energy, _sigma_fit, _d_fit = _popt
print(f"Fitted parameters:\n a = {_a_fit:.3f}, x0 = {photon_energy:.3f}, sigma = {_sigma_fit:.3f}, d = {_d_fit:.3f}")

# Plot
plt.figure(figsize=(8, 5))
plt.plot(spec_calib, specAverage, 'b.', label='Data')
plt.plot(spec_calib, _gaussian(spec_calib, *_popt), 'r-', label='Fit')
plt.xlabel('Photon Energy (eV)')
plt.ylabel('Intensity')
plt.title('Average Spectrum of Pulse')
plt.legend()
plt.grid(True)
plt.show()
wavelength = 12400/photon_energy  # get wavelength from the photon energy in eV

### Plotting the J4M Sum as a check, and then masking out the pixels. Special to L-10376 is masking out the Zn covered region.

In [ ]:
jungfrau_sum[jungfrau_sum>100*np.median(jungfrau_sum)]=0
plt.figure(figsize=(9,6))
pcm = plot_jungfrau(-y,x,jungfrau_sum,vmax=1e6)
plt.colorbar(pcm)
plt.show()

In [ ]:
masked_jungfrau_sum = np.copy(jungfrau_sum)
masked_jungfrau_sum[~(cmask*userMask)] = np.nan
plt.figure(figsize=(10,8))
pcm = plot_jungfrau(-y,x,masked_jungfrau_sum,vmax=30e5)
plt.colorbar(pcm)
plt.show()

### Writing a new version of the geometry calibration code:

In [ ]:
# Order of operations:

# Define fit function which outputs the scattering amplitude based on: A (scalar amplitude), X, Y, Z, phi, given the photon energy and theory pattern.
# The fit funtion should work like this:
# - First, a 1d therory pattern is loaded from the theory folder.
# - Then, it is made into a J4M shaped array



if fitPhi == 1:
    """Defining a function wrapper which will hold certain parameters constant"""
    def fitWrapper(xy,amplitude,X,Y,Z,phi):
        return geometryCalibration(xy,amplitude,X,Y,Z,phi,wavelength,isotropicTheory)
else:
    def fitWrapper(xy,amplitude,X,Y,Z):
        return geometryCalibration(xy,amplitude,X,Y,Z,0,wavelength,isotropicTheory)

def geometryCalibration(xy,amplitude,xCenter,yCenter,z,phi,wavelength,isotropicTheory):
    
    return scatteringAmplitude
    

In [ ]:
def theta_to_q(theta):
    return 4*np.pi*np.sin(theta/2.)/(wavelength)

def q_to_theta(q):
    return 2*np.arcsin((wavelength*q)/(4*np.pi))

def xyz_to_q(x,y,xcenter,ycenter,z):
    xcent = x-xcenter
    ycent = y-ycenter
    r_xy = np.sqrt(xcent**2+ycent**2)
    theta = np.arctan(r_xy/z0)  # theta is scattering angle
    return theta_to_q(theta)

def get_thomson_correction(x,y,z,phi0=0):   
    r_xy = np.sqrt(x**2+y**2)
    theta = np.arctan(r_xy/z)
    phi = np.arctan2(y,x)+phi0
    #polarization correction due to polarized x-rays interaction with electrons
    correction = (np.sin(phi)**2+np.cos(theta)**2*np.cos(phi)**2)
    return correction

def get_geometry_correction(x,y,z):
    r_xy = np.sqrt(x**2+y**2)
    theta = np.arctan(r_xy/z)
    # correction due to intensity falling off as 1/R**2
    R2 =  (np.cos(theta))**2 #  = (x**2+y**2+z**2)/z**2 
    # correction due to flux through pixel area falling off with increased angle
    A = np.cos(theta)
    return R2*A

def xy_to_phi(x,y):
    return np.arctan2(y,x) # + np.pi


def fitting_function(xy,xcenter,ycenter,z0,amplitude):
    x = np.ravel(xy[0])
    y = np.ravel(xy[1])
    xcent = x-xcenter
    ycent = y-ycenter
    r_xy = np.sqrt(xcent**2+ycent**2)
    theta = np.arctan(r_xy/z0)
    Q_abs = theta_to_q(theta)
    # ff = amplitude*(a*scattering_pattern_theory(Q_abs,I_pattern_theory,q_theory) - (1-a)*scattering_pattern_theory(Q_abs,N2_theory_pattern,q_theoryN2))
    ff = amplitude*scattering_pattern_theory(Q_abs,I_pattern_theory,q_theory) 
    thomson_correction = get_thomson_correction(xcent,ycent,z0)
    geometry_correction = get_geometry_correction(xcent,ycent,z0)
    ff = ff*thomson_correction*geometry_correction  # divide the theory by correction factors to match the experiment data
    return ff

def fitting_function_freephi(xy,xcenter,ycenter,z0,amplitude,phi0):
    x = np.ravel(xy[0])
    y = np.ravel(xy[1])
    xcent = x-xcenter
    ycent = y-ycenter
    r_xy = np.sqrt(xcent**2+ycent**2)
    theta = np.arctan(r_xy/z0)
    Q_abs = theta_to_q(theta)
    ff = amplitude*scattering_pattern_theory(Q_abs,I_pattern_theory,q_theory)

    thomson_correction = get_thomson_correction(xcent,ycent,z0,phi0=phi0)
    geometry_correction = get_geometry_correction(xcent,ycent,z0)
    ff = ff*thomson_correction*geometry_correction
    return ff
# This function safely outputs a scattering pattern from interpolated values even if the q range is outside of the defined q.
def scattering_pattern_theory(q1,I_pattern_theory,q_theory):
    output = np.zeros_like(q1) # Allocating the output array
    output[q1>np.max(q_theory)] = I_pattern_theory(np.max(q_theory)) # Handling the edges with flat interpolation
    output[q1<np.min(q_theory)] = I_pattern_theory(np.min(q_theory))
    # Doing the actual interpolation for the range where I_pattern_theory is defined
    output[(q1<=np.max(q_theory))&(q1>=np.min(q_theory))] = I_pattern_theory(q1[(q1<=np.max(q_theory))&(q1>=np.min(q_theory))])
    return output

In [ ]:
theory = np.loadtxt('./Theory/SF6_total.txt')
q_theory = theory[0]
I_theory = theory[1]
plt.figure()
plt.plot(q_theory,q_theory*I_theory)
plt.xlabel('q (inv. Ang.)',fontsize=12)
plt.ylabel('q*I(q)',fontsize=12)
plt.title('Theory SF6 I(q) Vs q',fontsize=15)
plt.show()

In [ ]:
from scipy.interpolate import interp1d  # interpolate function loaded
import scipy.io
from scipy.interpolate import InterpolatedUnivariateSpline

# theoryN2 = scipy.io.loadmat('./Theory_Curves/N2_IAM_pattern.mat')
# # # print(theory.shape)
# q_theoryN2 = np.squeeze(theoryN2['q']) # theory[0,:]  (azav[onshots,:]*Intensity_Average/dg2_on).sum(axis=0)
# I_theoryN2 = np.squeeze(theoryN2['N2_IAM'])

I_pattern_theory = interp1d(q_theory,I_theory) # interpolated scattering pattern theory
# N2_theory_pattern = interp1d(q_theoryN2,I_theoryN2)

#det_test = np.load('detector_efficiency113keV_SiBeAl.npz')
#detq = det_test['qbin_det']
#det_efficiency = det_test['detector_efficiency']

from scipy.interpolate import InterpolatedUnivariateSpline

#et_Spline = InterpolatedUnivariateSpline(detq, det_efficiency)

# This function safely outputs a scattering pattern from interpolated values even if the q range is outside of the defined q.
def scattering_pattern_theory(q1,I_pattern_theory,q_theory):
    output = np.zeros_like(q1) # Allocating the output array
    output[q1>np.max(q_theory)] = I_pattern_theory(np.max(q_theory)) # Handling the edges with flat interpolation
    output[q1<np.min(q_theory)] = I_pattern_theory(np.min(q_theory))
    # Doing the actual interpolation for the range where I_pattern_theory is defined
    output[(q1<=np.max(q_theory))&(q1>=np.min(q_theory))] = I_pattern_theory(q1[(q1<=np.max(q_theory))&(q1>=np.min(q_theory))])
    return output


qs = np.linspace(np.min(q_theory),np.max(q_theory),1000)  # Defining new q range
#Splined_det = det_Spline(qs)


plt.figure(figsize=(6,4))
plt.plot(qs, qs*scattering_pattern_theory(qs,I_pattern_theory,q_theory),label='SF6',color='r')
# plt.plot(qs, qs*scattering_pattern_theory(qs,N2_theory_pattern,q_theoryN2),label='N2',color='b')
plt.xlabel('q',fontsize=14)
plt.ylabel('q*I(q) ',fontsize=14)
plt.title(' Interpolated Theory SF6 I(q) Vs q', fontsize=16)
plt.legend(fontsize=14)
plt.show()

In [ ]:
x0 = 0
y0 = 0
z0 = 95000
phi=0
# x0, y0 = 0,0
# z0 = 82000.

amplitude = 449.0357305848682

goodmask = np.ones_like(x).astype(bool)
goodmask[np.sqrt(x**2+y**2)<7500]=0  # all pixels below 7500 are et to 0

xgood = x[goodmask]
ygood = y[goodmask]
xy = [xgood,ygood]

params1 = np.array([x0,y0,z0,amplitude])

# curve fit to find bext values
params1, covariances1 = curve_fit(fitting_function,xy,np.ravel(masked_jungfrau_sum),
                                p0=[x0,y0,z0,amplitude],
                                bounds=([-10000,-10000,30000,0],[10000,10000,150000,10000])) 

labels = ['x0','y0','z0','amp']

print('params1')
for i,label in enumerate(params1):
    print(labels[i],'=', float(params1[i])) # print values

In [ ]:
xy = [x,y]
x0 = params1[0]
y0 = params1[1]
z0 = params1[2]
amp = params1[3]
# phi = params1[4]
# manual additions
# x0 = 318.19686197368884
# y0 = -43.21838041984081
# z0 = 70092.61502970615

Q_new = xyz_to_q(x,y,x0,y0,z0)
# print(Q_new)
fitted_pattern = np.reshape(fitting_function(xy,*params1),(8,512,1024))  # Theory fit pattern to match the experiment

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
pcm = plot_jungfrau(-y,x,Q_new*fitted_pattern,vmin=0,vmax=np.max(fitted_pattern)/10)
plt.title('theory pattern')
plt.colorbar(pcm)
plt.subplot(1,2,2)
pcm = plot_jungfrau(-y,x,Q_new*jungfrau_sum,vmin=0,vmax=np.max(fitted_pattern)/10)
plt.title('Jungfrau sum')
plt.colorbar(pcm)
plt.subplots_adjust(hspace=0.25, wspace=0.25)
plt.show()

plt.figure(figsize=(5,4))
pcm = plot_jungfrau(-y,x,Q_new*(fitted_pattern-jungfrau_sum),vmin=0,vmax=np.max(fitted_pattern)/10)
plt.title('residue')
plt.colorbar(pcm)
plt.show()